# Charformer Turkish Base - Diacritics Restoration Experiment

**Goal:** Test if `orkungedik/charformer-turkish-base` can restore Turkish diacritics.

**Approach:**
1. Load the pre-trained model and run inference on ASCII Turkish text
2. Evaluate against our 2,942-entry ambiguous-lookup.tsv dataset
3. If promising, fine-tune with LoRA for diacritics restoration task

**Hardware:** Kaggle T4 x2 GPU

**Setup:** Upload `ambiguous-lookup.tsv` as a Kaggle dataset, then enable GPU accelerator.

## 0. Setup

In [ ]:
!pip install -q torch huggingface_hub

In [ ]:
import torch
import torch.nn as nn
import json
import math
from pathlib import Path
from huggingface_hub import hf_hub_download

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Download Model Files

In [ ]:
REPO_ID = "orkungedik/charformer-turkish-base"

vocab_path = hf_hub_download(REPO_ID, "vocab.json")
config_path = hf_hub_download(REPO_ID, "config.json")
model_path = hf_hub_download(REPO_ID, "model_msce.bin")

with open(vocab_path) as f:
    vocab = json.load(f)

with open(config_path) as f:
    config = json.load(f)

stoi = vocab["stoi"]
itos = {int(k): v for k, v in vocab["itos"].items()}

print(f"Config: {json.dumps(config, indent=2)}")
print(f"Vocab size: {len(stoi)} chars")
print(f"Special tokens: {[k for k in stoi if k.startswith('[')]}")

## 2. Reconstruct Model Architecture

The model is a custom Charformer (not standard HuggingFace), so we need to
reconstruct the architecture from the config and paper.

Architecture: GBST (Gradient-Based Subword Tokenization) + Transformer Encoder-Decoder

In [ ]:
# First, inspect the checkpoint to understand the exact architecture
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)

# Check what's in the checkpoint
if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint
else:
    state_dict = checkpoint.state_dict() if hasattr(checkpoint, 'state_dict') else checkpoint

print(f"\nTotal parameters: {sum(p.numel() for p in state_dict.values()):,}")
print(f"\nLayer names ({len(state_dict)} tensors):")
for name, param in sorted(state_dict.items()):
    print(f"  {name}: {list(param.shape)}")

## 3. Build Model from State Dict

Based on the layer names above, we reconstruct the model.
**Run cell 2 first, then update this cell based on actual layer names.**

In [ ]:
import torch.nn.functional as F

# === RMSNorm (no bias, matches state_dict ln*.weight only) ===
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight

# === GBST (matches gbst.block_projections, block_scoring_network, conv_layer) ===
class GBST(nn.Module):
    def __init__(self, d_model, max_subword, ds_rate):
        super().__init__()
        self.d_model = d_model
        self.max_subword = max_subword
        self.ds_rate = ds_rate

        self.block_projections = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(max_subword)
        ])

        score_in = d_model + max_subword  # 772
        score_mid = score_in // 2          # 386
        self.block_scoring_network = nn.Sequential(
            nn.Conv1d(score_in, score_mid, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(score_mid, 1, kernel_size=1)
        )

        self.conv_layer = nn.Conv1d(1, d_model, kernel_size=5, padding=2)

    def forward(self, x):
        B, T, D = x.shape

        block_reps = []
        for w in range(self.max_subword):
            block_reps.append(self.block_projections[w](x))
        stacked = torch.stack(block_reps, dim=-1)  # [B, T, D, max_subword]

        width_ind = torch.zeros(B, T, self.max_subword, device=x.device)
        for w in range(self.max_subword):
            width_ind[:, :, w] = 1.0 / (w + 1)

        score_input = torch.cat([x, width_ind], dim=-1).transpose(1, 2)
        scores = self.block_scoring_network(score_input)  # [B, 1, T]

        block_weights = torch.softmax(
            scores.transpose(1, 2).expand(-1, -1, self.max_subword), dim=-1
        )  # [B, T, max_subword]
        weighted = (stacked * block_weights.unsqueeze(2)).sum(dim=-1)

        # Conv layer processes score channel
        conv_out = self.conv_layer(scores)  # [B, D, T]
        weighted = weighted + conv_out.transpose(1, 2)

        return weighted[:, ::self.ds_rate, :]

# === Relative Bias (T5-style, matches rel_bias.relative_attention_bias.weight [128, 12]) ===
class RelativeBias(nn.Module):
    def __init__(self, num_buckets, num_heads):
        super().__init__()
        self.relative_attention_bias = nn.Embedding(num_buckets, num_heads)
        self.num_buckets = num_buckets

    def _bucket(self, rel_pos):
        n = -rel_pos
        nb = self.num_buckets // 2
        ret = (n < 0).long() * nb
        n = torch.abs(n)
        max_exact = nb // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float() / max_exact) / math.log(nb / max_exact) * (nb - max_exact)
        ).long()
        val_large = torch.min(val_large, torch.full_like(val_large, nb - 1))
        return ret + torch.where(is_small, n, val_large)

    def forward(self, seq_len, device):
        pos = torch.arange(seq_len, device=device)
        rel = pos.unsqueeze(0) - pos.unsqueeze(1)
        bias = self.relative_attention_bias(self._bucket(rel))
        return bias.permute(2, 0, 1).unsqueeze(0)

# === Structural Bias (matches struct_bias_layer.head_focus/tail_focus [1,12,1,1]) ===
class StructuralBias(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.head_focus = nn.Parameter(torch.zeros(1, num_heads, 1, 1))
        self.tail_focus = nn.Parameter(torch.zeros(1, num_heads, 1, 1))

    def forward(self, seq_len, device):
        p = torch.arange(seq_len, device=device, dtype=torch.float) / max(seq_len - 1, 1)
        return self.head_focus * (1 - p).view(1, 1, 1, -1) + self.tail_focus * p.view(1, 1, 1, -1)

# === Encoder Layer (matches enc_layers.{i}.*) ===
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, ff_dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model)
        )
        self.ln1 = RMSNorm(d_model)
        self.ln3 = RMSNorm(d_model)
        self.n_head = n_head
        self.head_dim = d_model // n_head

    def forward(self, x, attn_bias=None):
        h = self.ln1(x)
        B, T, D = h.shape
        q = self.q_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attn_bias is not None:
            attn = attn + attn_bias
        out = (F.softmax(attn, dim=-1) @ v).transpose(1, 2).reshape(B, T, D)
        x = x + self.alpha * self.o_proj(out)
        x = x + self.ffn(self.ln3(x))
        return x

# === Decoder Layer (matches dec_layers.{i}.* including cross_attn, gate) ===
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, ff_dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.register_buffer('last_gate_val', torch.tensor(0.0))
        # Self attention
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        # Cross attention
        self.cross_attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        # Gate
        self.gate_norm = RMSNorm(d_model)
        self.gate_proj = nn.Linear(d_model, 1)
        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model)
        )
        self.ln1 = RMSNorm(d_model)
        self.ln2 = RMSNorm(d_model)
        self.ln3 = RMSNorm(d_model)
        self.n_head = n_head
        self.head_dim = d_model // n_head

    def forward(self, x, memory, attn_bias=None, causal_mask=None):
        h = self.ln1(x)
        B, T, D = h.shape
        q = self.q_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attn_bias is not None:
            attn = attn + attn_bias
        if causal_mask is not None:
            attn = attn.masked_fill(causal_mask, float('-inf'))
        self_out = self.o_proj((F.softmax(attn, dim=-1) @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.alpha * self_out

        h = self.ln2(x)
        cross_out, _ = self.cross_attn(h, memory, memory)
        gate = torch.sigmoid(self.gate_proj(self.gate_norm(cross_out)))
        self.last_gate_val.fill_(gate.mean().item())
        x = x + gate * cross_out

        x = x + self.ffn(self.ln3(x))
        return x

# === MSCE (matches msce.gate, msce_projector, norm) ===
class MSCE(nn.Module):
    def __init__(self, d_model, morph_classes=4):
        super().__init__()
        self.msce_projector = nn.Sequential(
            nn.Linear(morph_classes, 192), nn.GELU(), nn.Linear(192, d_model)
        )
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, morph_logits):
        morph_feat = self.msce_projector(morph_logits)
        g = self.gate(torch.cat([x, morph_feat], dim=-1))
        return self.norm(x + g * morph_feat)

# === Full Charformer Model ===
class CharformerModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        D = config["d_model"]
        H = config["n_head"]
        N = config["num_layers"]
        V = config["vocab_size"]
        FF = D * 4

        self.emb = nn.Embedding(V, D)
        self.gbst = GBST(D, config["max_subword"], config["ds_rate"])
        self.enc_layers = nn.ModuleList([EncoderLayer(D, H, FF) for _ in range(N)])
        self.dec_layers = nn.ModuleList([DecoderLayer(D, H, FF) for _ in range(N)])
        self.rel_bias = RelativeBias(128, H)
        self.struct_bias_layer = StructuralBias(H)
        self.morph_head = nn.Linear(D, 4)
        self.msce = MSCE(D, 4)
        self.final_ln = RMSNorm(D)
        self.fc = nn.Linear(D, V, bias=False)

    def forward(self, src_ids, tgt_ids=None):
        x = self.gbst(self.emb(src_ids))
        B, T_enc, D = x.shape

        enc_bias = self.rel_bias(T_enc, x.device) + self.struct_bias_layer(T_enc, x.device)
        for layer in self.enc_layers:
            x = layer(x, attn_bias=enc_bias)
        memory = x

        dec_in = self.emb(tgt_ids if tgt_ids is not None else src_ids)
        T_dec = dec_in.shape[1]
        dec_bias = self.rel_bias(T_dec, dec_in.device)
        causal = torch.triu(torch.ones(T_dec, T_dec, device=dec_in.device, dtype=torch.bool), diagonal=1)
        causal = causal.unsqueeze(0).unsqueeze(0)

        for layer in self.dec_layers:
            dec_in = layer(dec_in, memory, attn_bias=dec_bias, causal_mask=causal)

        morph_logits = self.morph_head(dec_in)
        dec_in = self.msce(dec_in, morph_logits)
        return self.fc(self.final_ln(dec_in))

print("Charformer model architecture defined (matching state_dict).")

## 4. Load Weights & Test Inference

After matching the architecture to the state dict, load and test.

In [ ]:
# Helper: encode/decode text using vocab
def encode_text(text, stoi, max_len=512):
    ids = [stoi.get("[BOS]", 1)]
    for ch in text[:max_len - 2]:
        ids.append(stoi.get(ch, stoi.get("[UNK]", 3)))
    ids.append(stoi.get("[EOS]", 2))
    return ids

def decode_ids(ids, itos):
    chars = []
    for idx in ids:
        token = itos.get(idx, "")
        if token in ("[PAD]", "[BOS]", "[EOS]", "[UNK]", "[MASK]"):
            continue
        chars.append(token)
    return "".join(chars)

# Test encoding
test_ascii = "turkce cumle ornegi"
test_correct = "turkce cumle ornegi"  # Will be replaced with correct form
encoded = encode_text(test_ascii, stoi)
decoded = decode_ids(encoded, itos)
print(f"Input:   '{test_ascii}'")
print(f"Encoded: {encoded}")
print(f"Decoded: '{decoded}'")

In [ ]:
# Load model with strict mode to verify architecture match
model = CharformerModel(config)

result = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys:    {len(result.missing_keys)}")
print(f"Unexpected keys: {len(result.unexpected_keys)}")

if result.missing_keys:
    print(f"\nMissing: {result.missing_keys[:10]}...")
if result.unexpected_keys:
    print(f"\nUnexpected: {result.unexpected_keys[:10]}...")

total_params = sum(p.numel() for p in model.parameters())
loaded_params = sum(state_dict[k].numel() for k in state_dict if k in dict(model.named_parameters()) or k in dict(model.named_buffers()))
print(f"\nModel params:  {total_params:,}")
print(f"Loaded params: {loaded_params:,}")
print(f"Match rate:    {loaded_params/sum(p.numel() for p in state_dict.values())*100:.1f}%")

model = model.to(DEVICE).eval()
print(f"\nModel on {DEVICE}, eval mode.")

In [ ]:
# Inference test: feed ASCII Turkish, see what comes out
@torch.no_grad()
def predict_diacritics(text, model, stoi, itos, device=DEVICE):
    ids = encode_text(text, stoi)
    src = torch.tensor([ids], device=device)
    logits = model(src)
    pred_ids = logits.argmax(dim=-1)[0].tolist()
    return decode_ids(pred_ids, itos)

# Test samples
test_pairs = [
    ("turkce", "turkce"),
    ("acar", "acar/acar"),
    ("gorusuyoruz", "gorusuyoruz"),
    ("ogrenci", "ogrenci"),
    ("calisma", "calisma"),
    ("gunes", "gunes"),
    ("urun", "urun"),
]

print("Diacritics Restoration Test")
print("=" * 50)
for ascii_form, note in test_pairs:
    try:
        result = predict_diacritics(ascii_form, model, stoi, itos)
        print(f"  {ascii_form:20s} -> {result:20s} (expected note: {note})")
    except Exception as e:
        print(f"  {ascii_form:20s} -> ERROR: {e}")

## 5. Benchmark Against Lookup Table

Test the model against our 2,942-entry ambiguous-lookup.tsv dataset.

In [ ]:
# Load ambiguous-lookup.tsv from Kaggle dataset
# Upload the file as a Kaggle dataset first (Add Data > Upload)
import glob

# Auto-detect the dataset path
tsv_matches = glob.glob("/kaggle/input/**/ambiguous-lookup.tsv", recursive=True)
if tsv_matches:
    LOOKUP_PATH = tsv_matches[0]
    print(f"Found dataset at: {LOOKUP_PATH}")
else:
    LOOKUP_PATH = "/kaggle/input/turkish-diacritics/ambiguous-lookup.tsv"
    print(f"Using default path: {LOOKUP_PATH}")
    print("If not found, check your dataset name under /kaggle/input/")

def load_lookup(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                ascii_form = parts[0]
                # May have multiple correct forms separated by comma
                correct_forms = [x.strip() for x in parts[1].split(",")]
                pairs.append((ascii_form, correct_forms))
    return pairs

try:
    pairs = load_lookup(LOOKUP_PATH)
    print(f"Loaded {len(pairs)} ASCII -> diacritics pairs")
    print(f"Examples: {pairs[:5]}")
except FileNotFoundError:
    print(f"File not found: {LOOKUP_PATH}")
    print("Go to 'Add Data' > 'Upload' and add ambiguous-lookup.tsv as a dataset.")

In [ ]:
# Run benchmark
@torch.no_grad()
def benchmark(pairs, model, stoi, itos, device=DEVICE):
    correct = 0
    partial = 0
    total = len(pairs)
    errors = []

    for ascii_form, expected_forms in pairs:
        try:
            result = predict_diacritics(ascii_form, model, stoi, itos, device)
            result_clean = result.strip()

            if result_clean in expected_forms:
                correct += 1
            elif any(ef in result_clean or result_clean in ef for ef in expected_forms):
                partial += 1
                errors.append((ascii_form, expected_forms, result_clean, "partial"))
            else:
                errors.append((ascii_form, expected_forms, result_clean, "wrong"))
        except Exception as e:
            errors.append((ascii_form, expected_forms, str(e), "error"))

    print(f"\nBenchmark Results ({total} pairs)")
    print("=" * 50)
    print(f"  Exact match:   {correct}/{total} ({correct/total*100:.1f}%)")
    print(f"  Partial match: {partial}/{total} ({partial/total*100:.1f}%)")
    print(f"  Wrong/Error:   {len(errors)-partial}/{total} ({(len(errors)-partial)/total*100:.1f}%)")

    print(f"\nFirst 20 errors:")
    for ascii_f, expected, got, err_type in errors[:20]:
        print(f"  [{err_type}] {ascii_f} -> {got} (expected: {expected})")

    return correct, partial, errors

correct, partial, errors = benchmark(pairs, model, stoi, itos)

## 6. Fine-Tune Decision

Based on benchmark results above:

- **> 50% accuracy**: Model already understands Turkish character patterns. Fine-tune for diacritics.
- **20-50% accuracy**: Model has some understanding. Fine-tune may help significantly.
- **< 20% accuracy**: Model is too early-stage for this task. Revisit after stable release.

If fine-tuning is worth it, continue to the next section.

## 7. Fine-Tune for Diacritics Restoration (if benchmark is promising)

Task: seq2seq, ASCII input -> diacritics output

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DiacriticsDataset(Dataset):
    """Pairs of (ascii_text, correct_diacritics_text) for fine-tuning."""
    def __init__(self, pairs, stoi, max_len=128):
        self.pairs = pairs
        self.stoi = stoi
        self.max_len = max_len
        self.pad_id = stoi.get("[PAD]", 0)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ascii_form, correct_forms = self.pairs[idx]
        correct = correct_forms[0]  # Use first correct form

        src = encode_text(ascii_form, self.stoi, self.max_len)
        tgt = encode_text(correct, self.stoi, self.max_len)

        # Pad both to fixed max_len
        src = src + [self.pad_id] * (self.max_len - len(src))
        tgt = tgt + [self.pad_id] * (self.max_len - len(tgt))

        return torch.tensor(src), torch.tensor(tgt)

# Create dataset
try:
    dataset = DiacriticsDataset(pairs, stoi)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    print(f"Dataset: {len(dataset)} pairs")
    print(f"Batches: {len(loader)}")

    # Peek at one batch
    src_batch, tgt_batch = next(iter(loader))
    print(f"Batch shapes: src={src_batch.shape}, tgt={tgt_batch.shape}")
except NameError:
    print("Load the lookup table first (cell 5).")

In [ ]:
# Fine-tune loop
def fine_tune(model, loader, epochs=10, lr=1e-4, device=DEVICE):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore PAD

    for epoch in range(epochs):
        total_loss = 0
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)

            # Teacher forcing: feed target shifted right
            tgt_input = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]

            logits = model(src, tgt_input)

            # Reshape for loss
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_expected.reshape(B*T))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

    return model

model = fine_tune(model, loader, epochs=10, lr=1e-4)

In [ ]:
# Post fine-tune benchmark
model.eval()
correct_ft, partial_ft, errors_ft = benchmark(pairs, model, stoi, itos)
print(f"\nImprovement: {correct_ft - correct} more exact matches")

## 8. Save Fine-Tuned Model

In [ ]:
# Save fine-tuned weights to Kaggle output
OUTPUT_PATH = "/kaggle/working/charformer-turkish-diacritics.bin"
# torch.save(model.state_dict(), OUTPUT_PATH)
# print(f"Saved fine-tuned model to {OUTPUT_PATH}")
# print("Download from the 'Output' tab after notebook completes.")

## Notes

**Key uncertainty:** The model architecture reconstruction in cell 3 is a best guess
based on the Charformer paper + config.json. Cell 2 reveals actual layer names.
If they don't match, the `CharformerModel` class needs adjustment.

**Next steps after this notebook:**
1. If model works well: Open issue/PR on orkungedik/charformer-turkish-base
2. Share diacritics benchmark as a Turkish NLP community resource
3. Integrate as optional ML layer in turkish-diacritics hook